In [57]:
import requests
import pandas as pd

BASE_URL = "https://gamma-api.polymarket.com/markets"

def fetch_markets(limit=500):
    params = {
        "limit": limit,
        "offset": 0,
        "closed": "true"
    }
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json()

data = fetch_markets()
print(data[0].keys())
keys = list(data[0].keys())
for key in keys:
    print(key, data[0].get(key))
print(len(data))

dict_keys(['id', 'question', 'conditionId', 'slug', 'twitterCardImage', 'endDate', 'category', 'liquidity', 'image', 'icon', 'description', 'outcomes', 'outcomePrices', 'volume', 'active', 'marketType', 'closed', 'marketMakerAddress', 'updatedBy', 'createdAt', 'updatedAt', 'closedTime', 'mailchimpTag', 'archived', 'restricted', 'volumeNum', 'liquidityNum', 'endDateIso', 'hasReviewedDates', 'readyForCron', 'volume24hr', 'volume1wk', 'volume1mo', 'volume1yr', 'clobTokenIds', 'fpmmLive', 'volume1wkAmm', 'volume1moAmm', 'volume1yrAmm', 'volume1wkClob', 'volume1moClob', 'volume1yrClob', 'events', 'creator', 'ready', 'funded', 'cyom', 'competitive', 'pagerDutyNotificationEnabled', 'approved', 'rewardsMinSize', 'rewardsMaxSpread', 'spread', 'oneDayPriceChange', 'oneHourPriceChange', 'oneWeekPriceChange', 'oneMonthPriceChange', 'oneYearPriceChange', 'lastTradePrice', 'bestBid', 'bestAsk', 'clearBookOnStart', 'manualActivation', 'negRiskOther', 'umaResolutionStatuses', 'pendingDeployment', 'dep

In [58]:
import json

def is_binary_market(market):
    outcomes = market.get("outcomes", [])
    outcomes = json.loads(outcomes)
    return len(outcomes) == 2

def is_resolved(market):
    return market.get("closed") is True
    # outcomePricesStr = json.loads(market.get("outcomePrices"))
    # outcomePrices = [float(x) for x in outcomePricesStr]
    # return (outcomePrices == [0.0, 1.0] or outcomePrices == [1.0, 0.0])

def is_valid_market(market):
    return (
        is_binary_market(market) and
        is_resolved(market) and
        (market.get("volumeNum", 0) > 0)
    )

filtered = [m for m in data if is_valid_market(m)]

print("Filtered markets:", len(filtered))

Filtered markets: 471


In [55]:
from datetime import datetime

def get_outcome_label(market):
    outcomes = json.loads(market["outcomes"])
    prices = [float(x) for x in json.loads(market["outcomePrices"])]

    winner_index = prices.index(1.0)
    return 1 if outcomes[winner_index] == "Yes" else 0

def get_yes_token(market):
    outcomes = json.loads(market["outcomes"])
    token_ids = json.loads(market["clobTokenIds"])
    
    yes_index = outcomes.index("Yes")
    return token_ids[yes_index]

def parse_end_date(market):
    return datetime.fromisoformat(market["endDate"].replace("Z", "+00:00"))

In [56]:
yesno_data = []

for m in filtered:
    try:
        yesno_data.append({
            "id": m["id"],
            "question": m["question"],
            "end_time": parse_end_date(m),
            "yes_token": get_yes_token(m),
            "outcome": get_outcome_label(m),
            "volume": m["volumeNum"]
        })
    except Exception:
        continue

df = pd.DataFrame(yesno_data)

print(df.head())
print("Total yes/no markets:", len(df))

Empty DataFrame
Columns: []
Index: []
Total yes/no markets: 0


In [51]:
HISTORY_URL = "https://clob.polymarket.com/trades"

def fetch_price_history(token_id):
    params = {
        "token_id": token_id,
        "limit": 1000
    }
    response = requests.get(HISTORY_URL, params=params)
    response.raise_for_status()
    return response.json()

In [52]:
for i in range(50):
    sample = df.iloc[i]
    history = fetch_price_history(sample["yes_token"])
    print(i, history)

HTTPError: 401 Client Error: Unauthorized for url: https://clob.polymarket.com/trades?token_id=39050296324364445809641664684679872936491115626778324924525327416420800978175&limit=1000

In [1]:
import requests
import pandas as pd

BASE_URL = "https://data-api.polymarket.com"

def get_recent_trades(limit=20):
    url = f"{BASE_URL}/trades"
    params = {"limit": limit}

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def main():
    trades = get_recent_trades(limit=20)

    df = pd.DataFrame(trades)
    print("Columns:", df.columns.tolist())
    print()
    print(df.head())

if __name__ == "__main__":
    main()

Columns: ['proxyWallet', 'side', 'asset', 'conditionId', 'size', 'price', 'timestamp', 'title', 'slug', 'icon', 'eventSlug', 'outcome', 'outcomeIndex', 'name', 'pseudonym', 'bio', 'profileImage', 'profileImageOptimized', 'transactionHash']

                                  proxyWallet  side  \
0  0x26d80ff2cc865a20a19643124abe6d79a2ba6367  SELL   
1  0xc61a20898ffcd46ca706ce51c35f5aa49f32a891   BUY   
2  0xb7c41d24c25435f0a2b7a8bc3f15bae2fd279d90  SELL   
3  0xd0c6a037346f31cb46b5d24e6f5ec451c48ca952   BUY   
4  0x64f6b6e79033c1b3b435e17d6dfd94ec61ae1698   BUY   

                                               asset  \
0  8912428219832771874704677694238954074304253235...   
1  9144692024352645537158650806367908517869086282...   
2  9144692024352645537158650806367908517869086282...   
3  2909925545464781991941894605625118610601829520...   
4  8142734390593879711352479358774321743627836539...   

                                         conditionId       size     price  \
0  0x52c023485